In [15]:
import json
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Load Digits dataset
digits = load_digits()
X, y = digits.data, digits.target
mask = np.isin(y, [0, 1])
X, y = X[mask], y[mask]

# 2. PCA to 4 components
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X)

# 3. Scale between [0, pi/2] for optimal ZZFeatureMap rotation
scaler = MinMaxScaler(feature_range=(0, np.pi / 2))
X_scaled = scaler.fit_transform(X_pca)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)
print(f"Dataset preprocessed: {X_train.shape[0]} train, {X_test.shape[0]} test samples.")

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

# Re-initialize feature map with scaled dimensions
feature_map = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear')
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)

print("Re-evaluating Quantum Kernel Matrix with scaled data...")
matrix_train = quantum_kernel.evaluate(x_vec=X_train)
matrix_test = quantum_kernel.evaluate(x_vec=X_test, y_vec=X_train)
print("Done!")

Dataset preprocessed: 270 train, 90 test samples.
Re-evaluating Quantum Kernel Matrix with scaled data...


/tmp/ipykernel_7064/2446172789.py:32: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear')


Done!


In [10]:
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Processor A & B hardware coupling maps
coupling_a = CouplingMap([(0,1), (1,0), (1,2), (2,1), (1,3), (3,1), (3,4), (4,3)])
coupling_b = CouplingMap([(0,1),(1,0),(1,2),(2,1),(1,6),(6,1),(2,3),(3,2),(2,6),(6,2),
                          (2,5),(5,2),(3,4),(4,3),(4,5),(5,4),(5,6),(6,5),(6,7),(7,6),
                          (6,8),(8,6),(8,0),(0,8)])

basis_gates = ['rz', 'sx', 'x', 'cx']

def benchmark_processor(circuit, coupling_map, name):
    pm = generate_preset_pass_manager(optimization_level=3, coupling_map=coupling_map, basis_gates=basis_gates)
    transpiled = pm.run(circuit)
    ops = transpiled.count_ops()
    return {
        'processor': name,
        'depth': transpiled.depth(),
        'cx_count': int(ops.get('cx', 0)),
        'swap_count': int(ops.get('swap', 0)),
        'total_gates': int(sum(ops.values()))
    }

metrics_a = benchmark_processor(feature_map, coupling_a, "Processor A (5-qubit)")
metrics_b = benchmark_processor(feature_map, coupling_b, "Processor B (9-qubit)")

print("=" * 45)
print("Processor A Metrics:", metrics_a)
print("Processor B Metrics:", metrics_b)
print("=" * 45)

# Save JSON benchmark results into results directory
with open('../results/processor_a_transpilation.json', 'w') as f:
    json.dump(metrics_a, f, indent=4)

with open('../results/processor_b_transpilation.json', 'w') as f:
    json.dump(metrics_b, f, indent=4)

Processor A Metrics: {'processor': 'Processor A (5-qubit)', 'depth': 23, 'cx_count': 12, 'swap_count': 0, 'total_gates': 49}
Processor B Metrics: {'processor': 'Processor B (9-qubit)', 'depth': 23, 'cx_count': 12, 'swap_count': 0, 'total_gates': 49}


In [16]:
# 1. Quantum Support Vector Classifier
qsvc = SVC(kernel='precomputed')
qsvc.fit(matrix_train, y_train)
qsvc_preds = qsvc.predict(matrix_test)
q_acc = accuracy_score(y_test, qsvc_preds)

# 2. Classical Baseline 1: RBF Kernel SVM
rbf_svm = SVC(kernel='rbf')
rbf_svm.fit(X_train, y_train)
rbf_preds = rbf_svm.predict(X_test)
rbf_acc = accuracy_score(y_test, rbf_preds)

# 3. Classical Baseline 2: Linear SVM
lin_svm = SVC(kernel='linear')
lin_svm.fit(X_train, y_train)
lin_preds = lin_svm.predict(X_test)
lin_acc = accuracy_score(y_test, lin_preds)

print("=" * 45)
print(f"Quantum Kernel SVC Accuracy: {q_acc * 100:.2f}%")
print(f"Classical RBF SVM Accuracy:   {rbf_acc * 100:.2f}%")
print(f"Classical Linear SVM Accuracy: {lin_acc * 100:.2f}%")
print("=" * 45)

Quantum Kernel SVC Accuracy: 98.89%
Classical RBF SVM Accuracy:   100.00%
Classical Linear SVM Accuracy: 100.00%
